# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aabdullahhtar-create/flyrank-ML-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

**Phase:** Setup  
**Provisional lane:** Refresh / Content Opportunity Scoring

This notebook frames the decision before any modeling. The lane is provisional and can still be confirmed or changed through the end of Week 4.

## 1. My lane (or freestyle) and why

I am choosing the predefined **Refresh / Content Opportunity Scoring** lane. The practical problem is not simply to predict whether a page is “good” or “bad”; it is to help an SEO/content team decide **which pages deserve limited review and editing time first**. The starter dataset is a good fit because its unit is one pseudonymized content item and it contains observable search, traffic, freshness, position, CTR, and engagement signals. My provisional output will be a **ranked review queue** with a score and short reason codes. I will compare any learned method with a simple, transparent rule-based baseline rather than assuming that ML must be better.

In [1]:
# Load the starter data from the repo and verify its basic grain.
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

assert DATA_PATH.exists(), "Starter CSV not found. Run this notebook from the repo or work/notebooks directory."
df = pd.read_csv(DATA_PATH)

print(f"Starter dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")
print(f"Unique content items: {df['content_id'].nunique():,}")

Starter dataset: 30,000 rows x 44 columns
Pseudonymized clients: 32
Unique content items: 30,000


## 2. The question: decision, action, cost of a wrong call

**Research/search question.** Among eligible content pages, **which pages should an SEO/content editor review first for refresh or optimization, using only observable pre-decision signals, so limited editing time is focused on pages with evidence of decline or recoverable search opportunity?**

- **Unit of analysis:** one pseudonymized content item/page (`content_id`).
- **Decision:** which pages should enter the top of the review queue first.
- **Output:** a ranked page-level opportunity score plus interpretable reason codes. A later capstone version should prefer a future observed outcome for validation rather than the starter's current-window decline proxy.
- **Who acts and what action follows:** an SEO/content editor reviews the highest-ranked pages first and then decides whether a page needs a content refresh, CTR-focused optimization, deeper investigation, or no change. The score is triage support; it does not automatically edit or publish content.
- **Cost of a wrong recommendation:** a false positive can waste editor hours and may prompt an unnecessary change to a page that was performing acceptably. A false negative can leave a genuinely declining or recoverable page unreviewed, missing a chance to protect or regain search traffic. Because both errors matter, I should evaluate top-of-queue precision as well as how many meaningful opportunities the queue misses.
- **Why data/ML can help:** priority depends on several signals at once—visibility, position, freshness, CTR, traffic, engagement, and age—and their relationships may be too messy for one fixed hand-written rule. ML only earns a place if it produces a more useful and honestly validated ranking than a transparent baseline.

In [2]:
# Define the starter eligibility rule described in the lane guide.
eligible = (
    df.loc[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
      .drop_duplicates("content_id")
      .copy()
)

print(f"Eligible page-level decisions: {len(eligible):,}")
print("Decision grain check — duplicated content_id rows:", int(eligible["content_id"].duplicated().sum()))

Eligible page-level decisions: 30,000
Decision grain check — duplicated content_id rows: 0


## 3. Quick look at the data (2-3 real numbers)

The starter data gives a concrete reason to spend the next seven weeks on prioritization rather than treating every page equally:

1. **16,262 of 30,000 pages (54.21%)** are currently in the `down` trend bucket. This is a large review pool, so simply telling an editor “look at declining pages” would not prioritize the work enough.
2. **13,152 pages** are both in the `down` bucket **and have at least 100 impressions in the trailing 90 days**. That suggests many declining pages still have measurable search demand, making prioritization operationally relevant.
3. **9,759 pages** meet the starter rule for a **visible low-CTR page**: at least 500 impressions, a nonzero average position up to 20, and CTR below 0.5%. This is another sizeable opportunity group and shows that ranking can combine different types of evidence rather than using decline alone.

These numbers do **not** prove that refreshing those pages will improve traffic. They show that the starter snapshot contains enough measurable variation and enough candidate pages to justify testing a ranked review process.

In [3]:
# Reproduce three supporting numbers directly from the starter snapshot.
declining = eligible["trend_direction"].eq("down")
declining_with_demand = declining & eligible["impressions_90d"].ge(100)
low_ctr_visible = (
    eligible["impressions_90d"].ge(500)
    & eligible["avg_position"].gt(0)
    & eligible["avg_position"].le(20)
    & eligible["ctr"].lt(0.5)  # rates are stored as x100 percentages; 0.5 means 0.5%
)

summary = pd.DataFrame({
    "evidence": [
        "Pages in current 'down' trend bucket",
        "Down-trend pages with >=100 impressions_90d",
        "Visible pages with CTR <0.5% (starter rule)",
    ],
    "pages": [
        int(declining.sum()),
        int(declining_with_demand.sum()),
        int(low_ctr_visible.sum()),
    ],
    "share_of_eligible_pct": [
        declining.mean() * 100,
        declining_with_demand.mean() * 100,
        low_ctr_visible.mean() * 100,
    ],
})
summary["share_of_eligible_pct"] = summary["share_of_eligible_pct"].round(2)
summary

,evidence,pages,share_of_eligible_pct
0,Pages in current 'down' trend bucket,16262,54.21
1,Down-trend pages with >=100 impressions_90d,13152,43.84
2,Visible pages with CTR <0.5% (starter rule),9759,32.53


## 4. Careful words: what I can and can't claim

At this stage I can make **descriptive and decision-support claims** about this snapshot: for example, that certain observable page signals are associated with the current trend buckets, or that a ranking method concentrates more measured opportunities near the top of a review queue than a baseline. If I later build a future-window label from the warehouse data, I can test whether earlier signals have **predictive value for later observed outcomes** under an honest time split.

I **cannot** claim that a refresh *causes* traffic to increase, that the model understands Google's ranking system, or that a high score guarantees improvement. The starter `trend_direction` field is calculated from current-window trend information, so it is only a **proxy label** for this early framing exercise; `trend_direction` and `trend_pct` must not be used as model features when that proxy is the target. I will also treat `avg_position = 0` as missing/no-position data rather than rank zero, and I will remember that rate columns such as CTR are stored as ×100 percentages. Final recommendations should remain human-reviewed and should be described as prioritization evidence, not causal prescriptions.

In [4]:
# Sanity checks for claims and known starter-data gotchas.
checks = {
    "rows": len(df),
    "columns": df.shape[1],
    "avg_position_zero_means_no_data_rows": int(df["avg_position"].eq(0).sum()),
    "trend_direction_column_exists": "trend_direction" in df.columns,
    "trend_pct_column_exists": "trend_pct" in df.columns,
    "trend_pct_missing_rows": int(df["trend_pct"].isna().sum()),
}
checks

{'rows': 30000,
 'columns': 44,
 'avg_position_zero_means_no_data_rows': 1205,
 'trend_direction_column_exists': True,
 'trend_pct_column_exists': True,
 'trend_pct_missing_rows': 3388}

## Self-check

Before submission:

- [x] Every section above is filled with the required framing and supporting code.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, raw queries, or other private identifiers are shown.
- [x] My claims use careful words such as observed, measured, directional, proxy, predictive value, and decision-support.
- [x] I named the decision, the person who acts, the action, the unit of analysis, the output, and the cost of a wrong call.
- [x] I showed at least two real numbers from the starter dataset (three are shown).
- [x] I explained why this is a decision/ranking problem, not merely “train a model.”
- [ ] Commit this executed notebook to my public repo under `work/notebooks/w01_research_question.ipynb`, push it, and submit the repo URL on the assignment card.